In [3]:
import os
import pandas as pd

# 1. Define input and output paths
input_path = r"Large_files\services-2024-04.csv"
output_dir = r"Gefilterde datasets"
output_path = os.path.join(output_dir, "services-2024-04.csv")

# 2. Ensure the output folder exists
os.makedirs(output_dir, exist_ok=True)

# 3. Load the dataset
print("Loading dataset...")
df = pd.read_csv(input_path)

# 4. Filter out unwanted companies in 'Service:Company'
companies_to_exclude = ['DB', 'Eu Sleeper', 'Eurobahn', 'NMBS', 'NS Int']
df = df[~df['Service:Company'].astype(str).isin(companies_to_exclude)]

# 5. Filter out unwanted types in 'Service:Type'
types_to_exclude = ['Stopbus ipv trein', 'Taxibus ipv trein', 'Bus']
df = df[~df['Service:Type'].astype(str).isin(types_to_exclude)]

# 6. Keep only the specified dates in 'Service:Date' (using the correct YYYY-MM-DD format found in your data)
dates_to_keep = ['2024-04-20', '2024-04-27']
df = df[df['Service:Date'].astype(str).isin(dates_to_keep)]

# 7. Save the cleaned dataframe to the new folder
df.to_csv(output_path, index=False)

print(f"Filtering complete! Saved new file to:\n{output_path}")
print(f"Total remaining rows: {len(df)}")

Loading dataset...
Filtering complete! Saved new file to:
Gefilterde datasets\services-2024-04.csv
Total remaining rows: 98025


In [5]:
import os
import pandas as pd

# 1. Define input and output paths
input_path = r"Large_files\services-2024-04.csv"
output_dir = r"Large_files"
output_path = os.path.join(output_dir, "services-2024-04_cleaned_no_outlier.csv")

# 2. Ensure the output folder exists
os.makedirs(output_dir, exist_ok=True)

# 3. Load the unfiltered dataset
print("Loading unfiltered dataset...")
df = pd.read_csv(input_path)

# 4. Filter out unwanted companies in 'Service:Company'
companies_to_exclude = ['DB', 'Eu Sleeper', 'Eurobahn', 'NMBS', 'NS Int']
df = df[~df['Service:Company'].astype(str).isin(companies_to_exclude)]

# 5. Filter out unwanted types in 'Service:Type'
types_to_exclude = ['Stopbus ipv trein', 'Taxibus ipv trein', 'Bus']
df = df[~df['Service:Type'].astype(str).isin(types_to_exclude)]

# 6. Ensure 'Stop:Arrival delay' is numeric and dates are clean strings
df['Stop:Arrival delay'] = pd.to_numeric(df['Stop:Arrival delay'], errors='coerce').fillna(0)
df['Service:Date'] = df['Service:Date'].astype(str)

# 7. Group by date and calculate the sum of arrival delays for every day
daily_sums = df.groupby('Service:Date')['Stop:Arrival delay'].sum().reset_index()
daily_sums.rename(columns={'Stop:Arrival delay': 'Total_Arrival_Delay'}, inplace=True)

# 8. Drop the date with the anomaly (306.0 on 2024-04-02) and April 27th for the average calculation
# We can filter out 2024-04-02 directly and 2024-04-27 as requested before
dates_to_exclude_from_avg = ['2024-04-02', '2024-04-27']
daily_sums_filtered = daily_sums[~daily_sums['Service:Date'].isin(dates_to_exclude_from_avg)]

# Calculate the average of the remaining daily sums
average_delay = daily_sums_filtered['Total_Arrival_Delay'].mean()

# 9. (Optional) If you also want to remove the 306-row entirely from the saved dataset file:
df_cleaned_file = df[df['Service:Date'] != '2024-04-02']
df_cleaned_file.to_csv(output_path, index=False)
print(f"Cleaned dataset (without April 2nd anomaly) saved to:\n{output_path}\n")

# 10. Output the daily results as a Pandas DataFrame
print("--- Daily Total Arrival Delays (Pandas DataFrame) ---")
print(daily_sums.to_string(index=False))

print("\n" + "="*60)
print(f"Average of daily total delays (excluding 2024-04-02 and 2024-04-27): {average_delay:.2f}")
print("="*60)

Loading unfiltered dataset...
Cleaned dataset (without April 2nd anomaly) saved to:
Large_files\services-2024-04_cleaned_no_outlier.csv

--- Daily Total Arrival Delays (Pandas DataFrame) ---
Service:Date  Total_Arrival_Delay
  2024-04-01              17511.0
  2024-04-02                306.0
  2024-04-03              31518.0
  2024-04-04              51774.0
  2024-04-05              52081.0
  2024-04-06              43464.0
  2024-04-07              25189.0
  2024-04-08              44624.0
  2024-04-09              44872.0
  2024-04-10              40129.0
  2024-04-11              46010.0
  2024-04-12              45327.0
  2024-04-13              32610.0
  2024-04-14              31134.0
  2024-04-15              50389.0
  2024-04-16              49862.0
  2024-04-17              42221.0
  2024-04-18              46401.0
  2024-04-19              49772.0
  2024-04-20              45804.0
  2024-04-21              29699.0
  2024-04-22              40201.0
  2024-04-23              3